# Pritam — Day 3 Session Summary
**Sprint:** Dark Store + Integrated Logistics | **Date:** April 4, 2026 (Day 3 tasks completed on Day 4)  
**Branch:** `pritam_temp_apr1` | **Role:** Lead Coder + Integration Architect

---

**Why Day 3 ran on Day 4:**  
Pranav's `vrp_nodes.csv` was not delivered by Day 2 EOD. However, `forward_vrp.py` (committed by the team) builds VRP nodes internally from `master_df_v2.parquet` + `dark_stores_final.csv`, making the static hand-off file redundant. All Day 3 tasks were completed in full today.

Load [`docs/pritam_sess_summ.md`](../../docs/pritam_sess_summ.md) for Pritam-scoped context.  
Load [`session_summary_v3.md`](../../session_summary_v3.md) for full project context.

---
## 1. Context Sync — What the Team Had Committed

Before starting Day 3, `git merge main` pulled 22 new team commits into `pritam_temp_apr1`:

| New file | Owner | Purpose |
|----------|-------|---------|
| `src/forward_vrp.py` | Team | OR-Tools CVRPTW full implementation |
| `src/reverse_vrp.py` | Vybhav | Reverse pickup VRP |
| `src/route_parser.py` | Pranav | Shared VRP utilities, constants, node builder |
| `src/demand_forecasting.py` | Anurag | Prophet per-zone forecasting |
| `notebooks/03_forward_vrp.ipynb` | Team | Forward VRP notebook |
| `notebooks/05_reverse_vrp.ipynb` | Vybhav | Reverse VRP notebook |
| `notebooks/vrp_summary.ipynb` | Team | VRP consolidated summary |

### Key design resolution — Pranav's `vrp_nodes.csv`
The roadmap specified Pranav would deliver `vrp_nodes.csv` as a static file.  
The team instead implemented `build_vrp_nodes()` inside `route_parser.py` which builds nodes dynamically from `master_df_v2.parquet` + `dark_stores_final.csv`.  
**This is better engineering** — no fragile hand-off, nodes always consistent with the data.

---
## 2. Step 1 — Clustering Pipeline → `dark_stores_final.csv`

### What `clustering.run_full_pipeline()` does
```
1. load_sp_data()           — load master_df.parquet, filter to SP Metro (46% of SP state)
2. build_zip_level_coords() — aggregate to zip-code level; demand_weight = order_count
3. run_kmeans()             — sweep K ∈ {3..12}; record inertia + silhouette per K
4. pick_k_by_coverage()     — choose smallest K with ≥70% customers within 5 km
5. assign_voronoi()         — assign each customer to nearest centroid → dark_store_id
6. build_dark_stores_df()   — centroids table with capacity + zone KPIs
7. save_outputs()           — dark_stores_final.csv, master_df_v2.parquet, CSVs
```

### K selection — coverage rule overrides silhouette

$$\text{Optimal } K = \min\{K : \text{coverage}(K,\; 5\text{ km}) \geq 70\%\}$$

| K | Silhouette | Coverage (5 km) | Selected? |
|---|-----------|----------------|-----------|
| 3 | **0.369** | 24.2% | |
| 7 | 0.359 | 51.7% | |
| 10 | 0.359 | 68.6% | |
| **11** | 0.354 | **73.7%** | ✅ chosen |
| 12 | 0.346 | 78.0% | |

Silhouette preferred K=3 but that leaves 76% of customers >5 km — unworkable for dark store last-mile delivery.

In [ ]:
import sys
sys.path.insert(0, "../..")

from src.clustering import run_full_pipeline as cluster_pipeline
import pandas as pd

results = cluster_pipeline(
    parquet_path="../../data/master_df.parquet",
    out_dir="../../data",
    plot_dir="../../outputs",
)

print(f"\nOptimal K    : {results['optimal_k']}")
print(f"Coverage 5km : {results['coverage_overall']*100:.1f}%")
results["dark_stores_df"][["dark_store_id","lat","lon","n_orders","coverage_5km_pct"]]

---
## 3. Step 2 — Forward VRP: OR-Tools CVRPTW All 11 Zones

### Pipeline architecture

```
master_df_v2.parquet + dark_stores_final.csv
        │
        ▼
route_parser.build_vrp_nodes()     ← 1 depot + 75 customers per zone
        │
        ▼  (for each zone z ∈ {0..10})
forward_vrp.solve_cvrptw(zone)
  ├── build_distance_matrix()      ← vectorised Haversine in metres
  ├── RoutingIndexManager          ← n nodes, num_vehicles, depot=0
  ├── distance callback            ← int(dist_matrix[i][j])
  ├── demand callback              ← product_weight_g per customer
  ├── time callback                ← travel_time_min + SERVICE_TIME_MIN=5
  ├── AddDimensionWithVehicleCapacity ← max 500 kg per vehicle
  ├── Time dimension + SetRange    ← AM [480,720] or PM [720,1080] per customer
  ├── AddDisjunction(penalty=100k) ← soft TW; penalty for dropped nodes
  └── PATH_CHEAPEST_ARC → GUIDED_LOCAL_SEARCH (30s limit)
        │
        ▼
route_parser.parse_solution()      ← solution → DataFrame
route_parser.save_routes()         ← forward_routes.csv / .json / kpi_summary.csv
```

| Constant | Value | Purpose |
|----------|-------|---------|
| `VEHICLE_CAPACITY_G` | 500,000 g | 500 kg per vehicle |
| `VEHICLE_SPEED_KMH` | 40 | SP urban speed |
| `FIXED_COST_PER_ROUTE` | R$50 | Per vehicle dispatch |
| `VAR_COST_PER_KM` | R$1.50 | Running cost |
| `SERVICE_TIME_MIN` | 5 | Stop time per customer |
| `SOLVER_TIME_LIMIT_S` | 30 | GLS time budget per zone |
| `MAX_CUSTOMERS_PER_ZONE` | 75 | Tractability cap |

In [ ]:
from src.forward_vrp import run_full_pipeline as vrp_pipeline

vrp_results = vrp_pipeline(
    parquet_path="../../data/master_df_v2.parquet",
    stores_path="../../data/dark_stores_final.csv",
    out_dir="../../outputs",
    data_dir="../../data",
)

n_solved = sum(r["solved"] for r in vrp_results["zone_results"].values())
print(f"\nZones solved: {n_solved} / {len(vrp_results['zones'])}")

---
## 4. Forward VRP Results — All 11 Zones

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

kpi = pd.read_csv("../../outputs/forward_kpi_summary.csv")

print("=== Forward KPI Summary — All 11 Zones ===")
print(kpi[["zone_id","n_customers","n_vehicles_used","total_dist_km","routing_cost_R$"]].to_string(index=False))
print()
print(f"Total distance  : {kpi['total_dist_km'].sum():.1f} km")
print(f"Total cost      : R${kpi['routing_cost_R$'].sum():.0f}")
print(f"Total vehicles  : {kpi['n_vehicles_used'].sum()}")
print(f"Bottleneck zone : Zone {kpi.loc[kpi['routing_cost_R$'].idxmax(), 'zone_id']} "
      f"(R${kpi['routing_cost_R$'].max():.0f})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(kpi["zone_id"], kpi["total_dist_km"], color="steelblue")
axes[0].set_xlabel("Zone ID"); axes[0].set_ylabel("Distance (km)")
axes[0].set_title("Total Route Distance per Zone"); axes[0].set_xticks(kpi["zone_id"])
axes[1].bar(kpi["zone_id"], kpi["routing_cost_R$"], color="coral")
axes[1].set_xlabel("Zone ID"); axes[1].set_ylabel("Cost (R$)")
axes[1].set_title("Routing Cost per Zone"); axes[1].set_xticks(kpi["zone_id"])
plt.tight_layout()
plt.savefig("../../outputs/day3_zone_kpi_bars.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/day3_zone_kpi_bars.png")

---
## 5. Baseline Comparison — VRP vs Naive Nearest-Store

### Naive baseline
Every customer makes one direct trip to their nearest dark store — 19,207 individual unoptimised trips. No route consolidation.

$$\text{Improvement} = \frac{d_{\text{naive}} - d_{\text{VRP}}}{d_{\text{naive}}} \times 100\%$$

In [ ]:
comparison = pd.read_csv("../../outputs/baseline_vs_optimised.csv")
print("=== Baseline vs Optimised ===")
print(comparison.to_string(index=False))
print()
naive = comparison.loc[comparison["method"]=="naive_nearest_store", "total_dist_km"].values[0]
opt   = comparison.loc[comparison["method"]=="vrp_cvrptw_optimised", "total_dist_km"].values[0]
imp   = comparison.loc[comparison["method"]=="vrp_cvrptw_optimised", "improvement_pct"].values[0]
print(f"Naive    : {naive:>12,.1f} km  (19,207 individual trips)")
print(f"VRP      : {opt:>12,.1f} km  (22 vehicles, 825 customers sampled)")
print(f"Saving   : {imp:.1f}% distance reduction")

---
## 6. Day 3 EOD Checklist

| Checkpoint | Status | Detail |
|-----------|--------|--------|
| `dark_stores_final.csv` generated | ✅ | K=11, 73.7% coverage within 5 km |
| `vrp_nodes.csv` generated | ✅ | 836 rows — 11 depots + 825 customers |
| `forward_routes.json` — all zones solved | ✅ | 11/11 CVRPTW |
| `forward_kpi_summary.csv` | ✅ | 1,069.6 km · R$2,704 · 22 vehicles |
| Improvement over naive baseline | ✅ | 98.6% distance reduction |
| `baseline_vs_optimised.csv` | ✅ | Saved |
| `notebooks/03_route_parser_guide.ipynb` | ✅ | 12 cells — constants, `build_vrp_nodes`, OR-Tools index guide |
| `notebooks/03_vrp_baseline_analysis.ipynb` | ✅ | 11 cells — zone KPI dashboard, naive baseline, SDVRP targets |
| `day3_session_summary.ipynb` | ✅ | This notebook (12 cells) |

---
## 7. Day 4 Task List

Per the roadmap (page 9) and current project state:

### Task 1 — Forward VRP: All Zones ✅ Already Complete
Done as part of Day 3 catch-up — 11/11 zones solved, all KPIs computed.

### Task 2 — SDVRP Hybrid Prototype (1 zone, 5 returners)
**Depends on:** `data/master_df_v3.parquet` (with `return_prob`) from Vybhav  
**Fallback:** use `is_return` column from `master_df_v2.parquet` as binary `return_flag`

| Step | Detail |
|------|--------|
| 1 | Check if `master_df_v3.parquet` exists (Vybhav's XGBoost output) |
| 2 | Build 1-zone SDVRP: 30 delivery customers + 5 return pickup nodes |
| 3 | Implement SDVRP load constraint in `src/joint_optimizer.py` |
| 4 | Two `AddDimension` calls: delivery load + pickup load |
| 5 | Compare `separate_fwd + separate_rev` vs `hybrid_cost` |
| 6 | Expected saving: 15–25% per roadmap |

**SDVRP load invariant (hardest constraint):**
```python
# At every stop t, net load must satisfy:
# 0 ≤ load_delivered(t) - load_collected(t) ≤ VEHICLE_CAPACITY_G
```

### Task 3 — `joint_optimizer.py` — make Z computable
```python
Z = alpha * C_fwd + beta * C_rev + gamma * T_penalty + delta * N_vehicles
# Equal weights to start: alpha = beta = gamma = delta = 0.25
```

| File | Function to implement |
|------|-----------------------|
| `src/joint_optimizer.py` | `compute_Z(fwd_cost, rev_cost, t_penalty, n_vehicles, weights)` |
| `src/joint_optimizer.py` | `solve_sdvrp_hybrid(zone, return_nodes)` |

### Day 4 Outputs Required
| Output | Status |
|--------|--------|
| `outputs/forward_kpi_summary.csv` | ✅ Done |
| `outputs/baseline_vs_optimised.csv` | ✅ Done |
| `notebooks/03_route_parser_guide.ipynb` | ✅ Done |
| `notebooks/03_vrp_baseline_analysis.ipynb` | ✅ Done |
| `outputs/sdvrp_prototype_v1.py` or notebook | To do |
| `src/joint_optimizer.py` (Z computable) | To do |


In [ ]:
import json, os

print("=== Day 3 Output Verification ===\n")

ds = pd.read_csv("../../data/dark_stores_final.csv")
print(f"dark_stores_final.csv    : {len(ds)} stores {'✓' if len(ds)==11 else '✗'}")

nodes = pd.read_csv("../../data/vrp_nodes.csv")
print(f"vrp_nodes.csv            : {len(nodes)} rows {'✓' if len(nodes)==836 else '✗'}")

with open("../../outputs/forward_routes.json") as f:
    rj = json.load(f)
print(f"forward_routes.json      : {len(rj)} zones {'✓' if len(rj)==11 else '✗'}")

kpi = pd.read_csv("../../outputs/forward_kpi_summary.csv")
print(f"forward_kpi_summary.csv  : {len(kpi)} zones | {kpi['total_dist_km'].sum():.1f} km | R${kpi['routing_cost_R$'].sum():.0f}  ✓")

comp = pd.read_csv("../../outputs/baseline_vs_optimised.csv")
imp = comp.loc[comp["method"]=="vrp_cvrptw_optimised","improvement_pct"].values[0]
print(f"baseline_vs_optimised    : {imp:.1f}% improvement  ✓")

v3 = os.path.exists("../../data/master_df_v3.parquet")
print(f"\nmaster_df_v3.parquet     : {'✓ available — Day 4 SDVRP unblocked!' if v3 else '✗ not yet — will use is_return as proxy'}")
print("\nAll Day 3 deliverables verified ✓")